# Разработка прогнозной модели для оценки риска осложнений

**Тема:** Построение и сравнение моделей машинного обучения для прогнозирования в медицинской диагностике

**Цель занятия:** Научиться разрабатывать прогнозные модели для ИСППР, сравнивать различные алгоритмы и оценивать их не только по статистическим метрикам, но и с точки зрения клинической значимости (бизнес-метрик).

## Импорты

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve, precision_recall_curve)

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

## Настройка отображения

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style("whitegrid")

print("Все библиотеки успешно загружены!")

## Часть 1: Подготовка среды и генерация синтетических данных

Поскольку у нас нет доступа к реальным медицинским данным (и это хорошо с точки зрения этики), мы сгенерируем реалистичный синтетический датасет, основанный на реальных клинических закономерностях.

In [ ]:
np.random.seed(42)  # для воспроизводимости
n_patients = 1000

print("Генерируем синтетические данные о пациентах с внебольничной пневмонией...")

# Создаём данные
data = {
    # Демографические данные
    'age': np.random.normal(55, 18, n_patients).astype(int),  # возраст
    'gender': np.random.choice([0, 1], n_patients),  # 0 - жен, 1 - муж
    # Витальные признаки
    'temperature': np.random.normal(38.2, 1.2, n_patients),  # температура тела
    'heart_rate': np.random.normal(95, 20, n_patients).astype(int),  # пульс
    'respiratory_rate': np.random.normal(22, 6, n_patients).astype(int),  # частота дыхания
    'systolic_bp': np.random.normal(125, 18, n_patients).astype(int),  # систолическое АД
    'oxygen_saturation': np.random.normal(94, 3, n_patients),  # сатурация кислорода (%)
    # Лабораторные показатели
    'leukocytes': np.random.normal(12, 4, n_patients),  # лейкоциты (10^9/л)
    'neutrophils': np.random.normal(75, 12, n_patients),  # нейтрофилы (%)
    'lymphocytes': np.random.normal(18, 8, n_patients),  # лимфоциты (%)
    'crp': np.random.exponential(50, n_patients),  # С-реактивный белок (мг/л)
    'procalcitonin': np.random.exponential(0.5, n_patients),  # прокальцитонин (нг/мл)
    'creatinine': np.random.normal(85, 25, n_patients).astype(int),  # креатинин (мкмоль/л)
    'glucose': np.random.normal(6.1, 2.2, n_patients),  # глюкоза (ммоль/л)
    # Коморбидности (сопутствующие заболевания)
    'diabetes': np.random.choice([0, 1], n_patients, p=[0.75, 0.25]),  # диабет
    'chd': np.random.choice([0, 1], n_patients, p=[0.8, 0.2]),  # ИБС
    'ckd': np.random.choice([0, 1], n_patients, p=[0.85, 0.15]),  # ХБП
    'copd': np.random.choice([0, 1], n_patients, p=[0.82, 0.18]),  # ХОБЛ
}

# Преобразуем в DataFrame
df = pd.DataFrame(data)

# Ограничиваем значения физиологическими диапазонами
df['age'] = np.clip(df['age'], 18, 95)
df['temperature'] = np.clip(df['temperature'], 35.5, 41.0)
df['heart_rate'] = np.clip(df['heart_rate'], 40, 160)
df['respiratory_rate'] = np.clip(df['respiratory_rate'], 10, 40)
df['systolic_bp'] = np.clip(df['systolic_bp'], 70, 210)
df['oxygen_saturation'] = np.clip(df['oxygen_saturation'], 70, 100)
df['leukocytes'] = np.clip(df['leukocytes'], 2, 30)
df['neutrophils'] = np.clip(df['neutrophils'], 40, 95)
df['lymphocytes'] = np.clip(df['lymphocytes'], 5, 50)
df['crp'] = np.clip(df['crp'], 0, 300)
df['procalcitonin'] = np.clip(df['procalcitonin'], 0, 10)
df['creatinine'] = np.clip(df['creatinine'], 40, 300)
df['glucose'] = np.clip(df['glucose'], 3.5, 20)

print(f"Сгенерировано {n_patients} записей о пациентах")
print(f"   Признаков: {df.shape[1]}")
print("\nПервые 5 записей:")
print(df.head())
print("\nСтатистика по данным:")
print(df.describe())

## Часть 2: Генерация целевой переменной (риск осложнений)

Теперь самое важное — создадим целевую переменную на основе клинических закономерностей. В реальности это были бы данные из историй болезни, но мы смоделируем логику врача.

In [ ]:
print("Генерируем целевую переменную 'риск осложнений'...")

# Создаём "риск" на основе клинических факторов
# Чем выше значение, тем выше риск
risk_score = (
    # Возраст (чем старше, тем выше риск)
    (df['age'] - 50) / 30 * 0.15 +
    # Температура (высокая лихорадка)
    (df['temperature'] - 37) / 3 * 0.10 +
    # Тахикардия
    (df['heart_rate'] - 80) / 50 * 0.10 +
    # Тахипноэ (частое дыхание)
    (df['respiratory_rate'] - 16) / 15 * 0.10 +
    # Гипоксия (низкая сатурация)
    (100 - df['oxygen_saturation']) / 15 * 0.15 +
    # Лейкоцитоз
    (df['leukocytes'] - 10) / 15 * 0.08 +
    # Повышение СРБ
    np.log1p(df['crp']) / 5 * 0.08 +
    # Прокальцитонин (маркер бактериальной инфекции)
    np.log1p(df['procalcitonin']) * 0.08 +
    # Коморбидности
    df['diabetes'] * 0.04 +
    df['chd'] * 0.04 +
    df['ckd'] * 0.04 +
    df['copd'] * 0.04
)

# Добавляем случайный шум
risk_score += np.random.normal(0, 0.1, n_patients)

# Нормируем к диапазону 0-1
risk_score = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min())

# Создаём бинарную целевую переменную
# 1 - высокий риск (нужна госпитализация/интенсивная терапия)
# 0 - низкий риск (амбулаторное лечение)
threshold = 0.5
df['high_risk'] = (risk_score > threshold).astype(int)

# Проверяем баланс классов
print(f"\nРаспределение целевой переменной:")
print(f"   Низкий риск (0): {(df['high_risk'] == 0).sum()} пациентов ({(df['high_risk'] == 0).mean() * 100:.1f}%)")
print(f"   Высокий риск (1): {(df['high_risk'] == 1).sum()} пациентов ({(df['high_risk'] == 1).mean() * 100:.1f}%)")

# Визуализируем распределение
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма risk_score
axes[0].hist(risk_score, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Порог ({threshold})')
axes[0].set_xlabel('Риск-скор (непрерывный)')
axes[0].set_ylabel('Количество пациентов')
axes[0].set_title('Распределение непрерывного риска')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Столбчатая диаграмма для бинарной переменной
df['high_risk'].value_counts().plot(kind='bar', ax=axes[1], color=['green', 'red'], alpha=0.7)
axes[1].set_xlabel('Риск осложнений')
axes[1].set_ylabel('Количество пациентов')
axes[1].set_title('Распределение бинарной переменной')
axes[1].set_xticklabels(['Низкий риск (0)', 'Высокий риск (1)'], rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nПримеры пациентов с высоким и низким риском:")
# Покажем несколько примеров
high_risk_examples = df[df['high_risk'] == 1].head(3)
low_risk_examples = df[df['high_risk'] == 0].head(3)

print("\nПациенты с ВЫСОКИМ риском:")
for idx, row in high_risk_examples.iterrows():
    print(f"    Пациент {idx}: возраст {row['age']}, темп {row['temperature']:.1f}°C, "
          f"сатурация {row['oxygen_saturation']:.1f}%, СРБ {row['crp']:.1f}")

print("\nПациенты с НИЗКИМ риском:")
for idx, row in low_risk_examples.iterrows():
    print(f"    Пациент {idx}: возраст {row['age']}, темп {row['temperature']:.1f}°C, "
          f"сатурация {row['oxygen_saturation']:.1f}%, СРБ {row['crp']:.1f}")

## Часть 3: Предобработка данных и разделение на выборки

## Подготовка данных для машинного обучения

In [ ]:
print("🔧 ПОДГОТОВКА ДАННЫХ ДЛЯ ML\n")

# Отделяем признаки от целевой переменной
X = df.drop('high_risk', axis=1)
y = df['high_risk']

print(f"Размерность матрицы признаков X: {X.shape}")
print(f"Размерность целевой переменной y: {y.shape}")

# Разделяем на обучающую и тестовую выборки (70% / 30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\nРазделение на выборки:")
print(f"   Обучающая выборка: {X_train.shape[0]} пациентов ({len(X_train) / len(X) * 100:.1f}%)")
print(f"   Тестовая выборка: {X_test.shape[0]} пациентов ({len(X_test) / len(X) * 100:.1f}%)")
print(f"   Соотношение классов сохранено (stratify)")

# Масштабирование признаков (важно для логистической регрессии)
# Создаём списки числовых и категориальных признаков
numeric_features = ['age', 'temperature', 'heart_rate', 'respiratory_rate',
                    'systolic_bp', 'oxygen_saturation', 'leukocytes', 'neutrophils',
                    'lymphocytes', 'crp', 'procalcitonin', 'creatinine', 'glucose']
categorical_features = ['gender', 'diabetes', 'chd', 'ckd', 'copd']

# Масштабируем числовые признаки
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(X_test[numeric_features])

print(f"\nМасштабирование числовых признаков выполнено")
print(f"    Средние после масштабирования: {X_train_scaled[numeric_features].mean().mean():.2e}")
print(f"    Стандартные отклонения: {X_train_scaled[numeric_features].std().mean():.2f}")

# Сохраняем исходные и масштабированные данные
datasets = {
    'train': (X_train, y_train),
    'test': (X_test, y_test),
    'train_scaled': (X_train_scaled, y_train),
    'test_scaled': (X_test_scaled, y_test)
}

print("\nСтатистика по обучающей выборке (первые 5 строк):")
print(X_train.head())

## Часть 4: Обучение моделей

In [ ]:
print("ОБУЧЕНИЕ МОДЕЛЕЙ МАШИННОГО ОБУЧЕНИЯ\n")

# Создаём словарь с моделями
models = {
    'Логистическая регрессия': LogisticRegression(
        C=1.0, max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Случайный лес': RandomForestClassifier(
        n_estimators=100, max_depth=10, random_state=42, class_weight='balanced'
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss', use_label_encoder=False
    ),
    'CatBoost': CatBoostClassifier(
        iterations=100, depth=6, learning_rate=0.1,
        random_state=42, verbose=0, class_weights=[1, 1]
    )
}

# Для хранения результатов
results = {}
trained_models = {}

# Обучаем каждую модель
for name, model in models.items():
    print(f"\nОбучаем {name}...")
    # Выбираем данные (для логистической регрессии нужны масштабированные)
    if name == 'Логистическая регрессия':
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test
    
    # Обучение
    model.fit(X_tr, y_train)
    
    # Предсказания
    y_pred = model.predict(X_te)
    y_pred_proba = model.predict_proba(X_te)[:, 1]
    
    # Сохраняем модель и результаты
    trained_models[name] = model
    results[name] = {
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'X_test': X_te
    }
    print(f"    Обучение завершено")

## Часть 5: Оценка моделей по статистическим метрикам

In [ ]:
print("СРАВНЕНИЕ МОДЕЛЕЙ ПО СТАТИСТИЧЕСКИМ МЕТРИКАМ\n")

# Создаём DataFrame для результатов
metrics_df = pd.DataFrame()

for name, result in results.items():
    y_pred = result['y_pred']
    y_pred_proba = result['y_pred_proba']
    
    # Вычисляем метрики
    metrics = {
        'Модель': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    
    # Добавляем в DataFrame
    metrics_df = pd.concat([metrics_df, pd.DataFrame([metrics])], ignore_index=True)

# Округляем до 3 знаков
metrics_df[['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']] = \
    metrics_df[['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']].round(3)

print("Таблица сравнения метрик:")
print(metrics_df.to_string(index=False))

# Находим лучшую модель по F1-score
best_model_f1 = metrics_df.loc[metrics_df['F1-score'].idxmax(), 'Модель']
best_f1 = metrics_df['F1-score'].max()
print(f"\nЛучшая модель по F1-score: {best_model_f1} ({best_f1:.3f})")

# Визуализация сравнения метрик
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Сравнение всех метрик (тепловая карта)
metrics_for_plot = metrics_df.set_index('Модель')
sns.heatmap(metrics_for_plot, annot=True, cmap='YlGnBu', fmt='.3f', ax=axes[0])
axes[0].set_title('Сравнение метрик моделей')

# График 2: Сравнение Precision-Recall
for name, result in results.items():
    y_pred_proba = result['y_pred_proba']
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    axes[1].plot(recall, precision, linewidth=2, label=f'{name} (AUC={metrics_for_plot.loc[name, \"ROC-AUC\"]:.3f})')

axes[1].set_xlabel('Recall (Чувствительность)')
axes[1].set_ylabel('Precision (Точность)')
axes[1].set_title('Precision-Recall кривые')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Часть 6: Интерпретация моделей (важность признаков)

In [ ]:
print("АНАЛИЗ ВАЖНОСТИ ПРИЗНАКОВ\n")

# Для каждой модели получаем важность признаков
feature_importance = pd.DataFrame({'Признак': X.columns})

# Логистическая регрессия (коэффициенты)
lr_model = trained_models['Логистическая регрессия']
feature_importance['Лог. регрессия'] = np.abs(lr_model.coef_[0])

# Случайный лес
rf_model = trained_models['Случайный лес']
feature_importance['Случайный лес'] = rf_model.feature_importances_

# XGBoost
xgb_model = trained_models['XGBoost']
feature_importance['XGBoost'] = xgb_model.feature_importances_

# CatBoost
cb_model = trained_models['CatBoost']
feature_importance['CatBoost'] = cb_model.feature_importances_

# Нормируем для сравнения
for col in ['Лог. регрессия', 'Случайный лес', 'XGBoost', 'CatBoost']:
    feature_importance[col] = feature_importance[col] / feature_importance[col].max()

print("Топ-10 важных признаков (усреднённый рейтинг):")
# Вычисляем среднюю важность
feature_importance['Средняя'] = feature_importance[['Лог. регрессия', 'Случайный лес', 'XGBoost', 'CatBoost']].mean(axis=1)
top_features = feature_importance.nlargest(10, 'Средняя')
print(top_features[['Признак', 'Средняя']].to_string(index=False))

# Визуализация
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
models_for_plot = ['Лог. регрессия', 'Случайный лес', 'XGBoost', 'CatBoost']

for idx, model_name in enumerate(models_for_plot):
    ax = axes[idx // 2, idx % 2]
    top_n = feature_importance.nlargest(8, model_name)
    ax.barh(top_n['Признак'], top_n[model_name], color='skyblue')
    ax.set_xlabel('Относительная важность')
    ax.set_title(f'{model_name}')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nКЛИНИЧЕСКАЯ ИНТЕРПРЕТАЦИЯ:")
print("   Сатурация кислорода (SpO2) - ключевой предиктор риска")
print("   Возраст и температура также важны")
print("   Маркеры воспаления (СРБ, прокальцитонин) подтверждают тяжесть")
print("   Коморбидности повышают риск, но уступают острым показателям")

## Часть 7: Бизнес-метрики (стоимость ошибок в медицине)

Это самая важная часть! В медицине цена ошибки разная:
1. **Ложноотрицательный результат (FN)**: пациент с высоким риском отправлен домой → может развиться тяжёлое осложнение, вплоть до летального исхода. Цена катастрофическая.
2. **Ложноположительный результат (FP)**: здоровый пациент госпитализирован → лишние затраты, неудобства, но безопасно.

In [ ]:
print("АНАЛИЗ БИЗНЕС-МЕТРИК (СТОИМОСТЬ ОШИБОК)\n")

# Определяем стоимость ошибок (в условных единицах)
# Это модель "цена решения", которую мы обсуждали на лекции
costs = {
    'true_negative': 0,  # правильный низкий риск - всё хорошо
    'true_positive': 0,  # правильный высокий риск - лечение назначено верно
    'false_positive': 1000,  # ложная тревога - лишняя госпитализация (~1000 у.е.)
    'false_negative': 10000  # пропущенный риск - тяжелые осложнения, реанимация (~10000 у.е.)
}

print("СТОИМОСТЬ ОШИБОК (в условных единицах):")
print(f"   Ложноположительный (FP): {costs['false_positive']} у.е. (лишняя госпитализация)")
print(f"   Ложноотрицательный (FN): {costs['false_negative']} у.е. (осложнения, реанимация)")
print(f"   Правильные решения: 0 у.е.\n")

# Функция для расчёта общей стоимости
def calculate_total_cost(y_true, y_pred, costs):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    total_cost = (tn * costs['true_negative'] +
                  fp * costs['false_positive'] +
                  fn * costs['false_negative'] +
                  tp * costs['true_positive'])
    return total_cost, tn, fp, fn, tp

# Рассчитываем стоимость для каждой модели
business_results = []
for name, result in results.items():
    y_pred = result['y_pred']
    total_cost, tn, fp, fn, tp = calculate_total_cost(y_test, y_pred, costs)
    business_results.append({
        'Модель': name,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
        'Всего ошибок': fp + fn,
        'Общая стоимость': total_cost,
        'Стоимость на пациента': total_cost / len(y_test)
    })

# Создаём DataFrame
business_df = pd.DataFrame(business_results)
business_df = business_df.sort_values('Общая стоимость')

print("РЕЗУЛЬТАТЫ С ТОЧКИ ЗРЕНИЯ БИЗНЕС-МЕТРИК:")
print(business_df.to_string(index=False))

# Находим лучшую модель по стоимости
best_business_model = business_df.iloc[0]['Модель']
best_business_cost = business_df.iloc[0]['Общая стоимость']
print(f"\nЛучшая модель с точки зрения стоимости ошибок: {best_business_model}")
print(f"   Общая стоимость на тестовой выборке: {best_business_cost:,.0f} у.е.")
print(f"   Средняя стоимость на пациента: {business_df.iloc[0]['Стоимость на пациента']:.0f} у.е.")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Сравнение общей стоимости
colors = ['skyblue', 'lightgreen', 'lightcoral', 'plum']
bars = axes[0].bar(business_df['Модель'], business_df['Общая стоимость'], color=colors, alpha=0.7)
axes[0].set_xlabel('Модель')
axes[0].set_ylabel('Общая стоимость (у.е.)')
axes[0].set_title('Сравнение общей стоимости ошибок')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Добавляем значения на столбцы
for bar, cost in zip(bars, business_df['Общая стоимость']):
    axes[0].text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 500,
                f'{cost:,.0f}', ha='center', va='bottom', fontsize=9)

# График 2: Матрица ошибок для лучшей модели
best_model_name = business_df.iloc[0]['Модель']
best_result = results[best_model_name]
cm = confusion_matrix(y_test, best_result['y_pred'])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Низкий риск', 'Высокий риск'],
            yticklabels=['Низкий риск', 'Высокий риск'])
axes[1].set_xlabel('Предсказание')
axes[1].set_ylabel('Фактическое')
axes[1].set_title(f'Матрица ошибок: {best_model_name}')

plt.tight_layout()
plt.show()